# BOCK → Knowledge Graph (KG) Builder

**Source:** BOCK `kg_rels.csv` + node CSV files  
**Species:** *Homo sapiens*

## What this notebook does

1. Loads BOCK node metadata files to build entity ID → label and ID → name lookup dictionaries.
2. Loads the full BOCK edge file (`kg_rels.csv`), filters out OLI (oligopeptide) nodes, maps entity types and builds Relation strings.
3. For each of the **12 active relation types**, annotates Head/Tail nodes with canonical identifiers and names, then exports one CSV per relation type.

## Pipeline overview per relation type

| Relation | Head ID | Tail ID | Head annotation | Tail annotation |
|---|---|---|---|---|
| Gene_Gene | ENSG→NCBI Symbol | ENSG→NCBI Symbol | NCBI description | NCBI description |
| Gene_Phenotype | ENSG→NCBI Symbol | HPO ID | NCBI description | HPO name |
| Gene_BiologicalProcess | ENSG→NCBI Symbol | GO ID | NCBI description | GO term name |
| Gene_CellularComponent | ENSG→NCBI Symbol | GO ID | NCBI description | GO term name |
| Gene_MolecularFunction | ENSG→NCBI Symbol | GO ID | NCBI description | GO term name |
| Gene_ProteinComplex | ENSG→NCBI Symbol | complex ID | NCBI description | — |
| BiologicalProcess_BiologicalProcess | GO ID | GO ID | GO term name | GO term name |
| MolecularFunction_MolecularFunction | GO ID | GO ID | GO term name | GO term name |
| CellularComponent_CellularComponent | GO ID | GO ID | GO term name | GO term name |
| Phenotype_Phenotype | HPO ID | HPO ID | HPO name | HPO name |
| Disease_Phenotype | OMIM → DOID | HPO ID | — | HPO name |

## Dropped (commented out in original)
- `Gene_ProteinFamily` — commented out
- `Gene_ProteinDomain` — commented out

## Output files
```
BOCK/   Gene_Gene_BOCK.csv
BOCK/   Gene_Phenotype_BOCK.csv
BOCK/   Gene_BiologicalProcess_BOCK.csv
BOCK/   Gene_CellularComponent_BOCK.csv
BOCK/   Gene_MolecularFunction_BOCK.csv
BOCK/   Gene_ProteinComplex_BOCK.csv
BOCK/   BiologicalProcess_BiologicalProcess_BOCK.csv
BOCK/   MolecularFunction_MolecularFunction_BOCK.csv
BOCK/   CellularComponent_CellularComponent_BOCK.csv
BOCK/   Phenotype_Phenotype_BOCK.csv
BOCK/   Disease_Phenotype_BOCK.csv
```

---
## 0 · Configuration — edit ONLY these two lines

In [1]:
import os
import re
import numpy as np
import pandas as pd
from glob import glob

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/BOCK/"

# ── Derived input paths (do not edit below this line) ────────────────────────
ENS2NCBI_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
NCBI_HUMAN_PATH = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
HPO_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/hpo/HPO.csv")
DO_PATH         = os.path.join(BASE_PATH, "databases_for_mapping/DO/All_DO_ref_files.csv")
GO_PATH         = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_GO_ID_NAME.csv")

# BOCK node and edge files
BOCK_NODES_PATH = os.path.join(BASE_PATH, "BOCK/")
BOCK_RELS_PATH  = os.path.join(BASE_PATH, "BOCK/kg_rels.csv")

os.makedirs(OUT_PATH, exist_ok=True)

print("Paths configured successfully.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output dir  : {OUT_PATH}")

Paths configured successfully.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output dir  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/BOCK/


---
## 1 · Load reference dictionaries

In [5]:
# ── 1a. ENSEMBL ↔ NCBI gene symbol crosswalk ─────────────────────────────────
# Note: NCBI_2ENS__dict maps ENS_ID -> Symbol (initial_alias -> name)
# It is loaded for completeness but not actively used for lookup in this notebook
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['initial_alias'], ENS_2NCBI['name']))
del ENS_2NCBI
print(f"ENS -> Symbol: {len(NCBI_2ENS__dict):,}")

ENS -> Symbol: 42,530


In [6]:
# ── 1b. NCBI Human gene_info ──────────────────────────────────────────────────
NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['Symbol']))
print(f"NCBI Human genes: {len(NCBI_ALL_GENEname_dict):,}")

NCBI Human genes: 193,352


In [7]:
# ── 1c. Gene synonym map — built ONCE globally ────────────────────────────────
# With ~10 calls across Gene sections, that's 10x unnecessary full NCBI_ALL_GENE scans.
# Built here once; all helper calls reuse this shared dict.
synonym_map = {}
for _, row in NCBI_ALL_GENE.iterrows():
    for syn in row['Synonyms'].split('|'):
        synonym_map[syn.strip()] = row['Symbol']
print(f"Gene synonym map: {len(synonym_map):,} aliases")

Gene synonym map: 69,564 aliases


In [11]:
# ── 1d. HPO phenotype: ID -> name ─────────────────────────────────────────────
phenotype = pd.read_csv(HPO_PATH, sep=',')
phenotype = phenotype[phenotype['id'].str.startswith('HP')]
phenotype_dict = dict(zip(phenotype['id'], phenotype['name']))
print(f"HPO entries: {len(phenotype_dict):,}")

HPO entries: 19,483


In [12]:
# ── 1e. Disease Ontology — build OMIM -> DOID lookup ──────────────────────────
# BOCK disease nodes use OMIM IDs (mim:XXXXXX format).
# We resolve them to DOID via the DO xrefs column.
combined_df = pd.read_csv(DO_PATH)
combined_df = combined_df[~combined_df['xrefs'].isna()]

def extract_omim_lower(xrefs):
    """Extract 'mim:XXXXXX' (lowercase) from DO xrefs string for BOCK matching."""
    if pd.isna(xrefs):
        return None
    match = re.search(r'MIM:(\d+)', xrefs)
    return f'mim:{match.group(1)}' if match else None

combined_df['OMIM'] = combined_df['xrefs'].apply(extract_omim_lower)
OMIM_df = combined_df[~combined_df['OMIM'].isna()].copy()
OMIM_dict = dict(zip(OMIM_df['OMIM'], OMIM_df['id']))  # {mim:XXXXXX: DOID:XXXXXXX}
print(f"OMIM -> DOID: {len(OMIM_dict):,} entries")

OMIM -> DOID: 5,329 entries


In [13]:
# ── 1f. GO term ID -> name ─────────────────────────────────────────────────────
GO = pd.read_csv(GO_PATH)
GO_dict = dict(zip(GO['id'], GO['name']))
print(f"GO terms: {len(GO_dict):,}")

GO terms: 47,995


---
## 2 · Load BOCK node metadata

Node files provide the entity ID → type and ID → name lookups used throughout.

In [14]:
# Node CSV files containing entity metadata: id:ID, uri, name, abbrevType, :LABEL
node_files = [
    "kg_nodes_disease.csv",
    "kg_nodes_gene.csv",
    "kg_nodes_protein_complex.csv",
    "kg_nodes_domain_family.csv",
    "kg_nodes_go.csv",
    "kg_nodes_phenotype.csv"
]

columns_needed = ["id:ID", "uri", "name", "abbrevType", ":LABEL"]
dfs = []
for fname in node_files:
    fpath = os.path.join(BOCK_NODES_PATH, fname)
    df = pd.read_csv(fpath, sep='\t', usecols=columns_needed, dtype=str)
    dfs.append(df)

BOCK_Nodes_labels = pd.concat(dfs, ignore_index=True)

# Build lookup dicts: entity ID -> node type and entity ID -> human-readable name
BOCK_Nodes_labels_dict = dict(zip(BOCK_Nodes_labels['id:ID'], BOCK_Nodes_labels[':LABEL']))
BOCK_Nodes_names_dict  = dict(zip(BOCK_Nodes_labels['id:ID'], BOCK_Nodes_labels['name']))

print(f"BOCK nodes loaded: {len(BOCK_Nodes_labels):,}")
print("Node types present:", set(BOCK_Nodes_labels[':LABEL'].dropna()))
BOCK_Nodes_labels.head(3)

GP-KG nodes loaded: 157,846
Node types present: {'MolecularFunction', 'ProteinDomain', 'ProteinComplex', 'Phenotype', 'CellularComponent', 'BiologicalProcess', 'Disease', 'Gene', 'ProteinFamily'}


,id:ID,uri,name,abbrevType,:LABEL
0,mim:619340,http://identifiers.org/mim/619340,Developmental and epileptic encephalopathy 96,D,Disease
1,mim:609153,http://identifiers.org/mim/609153,"Pseudohyperkalemia, familial, 2, due to red ce...",D,Disease
2,mim:224550,http://identifiers.org/mim/224550,Dystonia with ringbinden,D,Disease


---
## 3 · Load BOCK edges and build base triple table

In [15]:
# ── 3a. Load full edge file and filter ────────────────────────────────────────
BOCK = pd.read_csv(BOCK_RELS_PATH, sep='\t')

# Remove OLI (oligopeptide) nodes — not included in EvoAge KG scope
before = len(BOCK)
BOCK = BOCK[~BOCK[':START_ID'].str.startswith('OLI')]
BOCK = BOCK[~BOCK[':END_ID'].str.startswith('OLI')]
print(f"After OLI filter: {len(BOCK):,} (removed {before - len(BOCK):,})")

# Rename to KG schema columns
BOCK.rename(columns={
    ':START_ID': 'Head',
    ':END_ID':   'Tail',
    ':TYPE':     'Relation_type'
}, inplace=True)

BOCK['KG_Source']  = 'BOCK'
BOCK['Head_type']  = BOCK['Head'].map(BOCK_Nodes_labels_dict)
BOCK['Tail_type']  = BOCK['Tail'].map(BOCK_Nodes_labels_dict)
BOCK['Relation']   = BOCK['Head_type'] + '_' + BOCK['Tail_type']

# Select schema columns
desired_cols = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type",
                "Source","KG_Source","Head_ID","Head_Detail_name","Head_ENS",
                "Tail_name","Tail_DOID_Name","Tail_ENS","Head_ID_IS","Tail_ID_IS",
                "Head_Orignal","Tail_Orignal"]
BOCK = BOCK[[c for c in desired_cols if c in BOCK.columns]]

# Deduplicate on the triple
BOCK.drop_duplicates(subset=["Head","Relation","Tail"], inplace=True)

print(f"Total unique triples: {len(BOCK):,}")
print("\nRelation distribution:")
print(BOCK['Relation'].value_counts())

/tmp/ipykernel_1517944/2951291573.py:2: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  GPKG = pd.read_csv(GPKG_RELS_PATH, sep='\t')


After OLI filter: 2,655,209 (removed 3,873)
Total unique triples: 2,638,941

Relation distribution:
Relation
Gene_Gene                              1838742
Disease_Phenotype                       232881
Gene_Phenotype                          209416
Gene_BiologicalProcess                   93676
Gene_CellularComponent                   58432
Gene_ProteinFamily                       45468
Gene_MolecularFunction                   43331
Gene_ProteinDomain                       41330
BiologicalProcess_BiologicalProcess      33102
Phenotype_Phenotype                      16000
Gene_ProteinComplex                      14531
MolecularFunction_MolecularFunction      11239
CellularComponent_CellularComponent        793
Name: count, dtype: int64


In [16]:
# ── 3b. Check for unmapped entity types ───────────────────────────────────────
# Rows where Head_type or Tail_type is NaN mean the entity ID was not in node files
unmapped_head = BOCK[BOCK['Head_type'].isna()]
unmapped_tail = BOCK[BOCK['Tail_type'].isna()]
print(f"Rows with unmapped Head_type: {len(unmapped_head):,}")
print(f"Rows with unmapped Tail_type: {len(unmapped_tail):,}")

Rows with unmapped Head_type: 0
Rows with unmapped Tail_type: 0


---
## 4 · Helper functions

In [17]:
def resolve_BOCK_name(df, col):
    """
    Resolve a BOCK node ID to its human-readable name via BOCK_Nodes_names_dict.
    Drops rows where the name cannot be resolved (entity not in node files).
    Swaps: resolved name -> col, original ID -> {col}_Orignal (kept for audit).
    """
    df = df.copy()
    orig_col = f'{col}_Orignal'
    df[orig_col] = df[col].map(BOCK_Nodes_names_dict)
    before = len(df)
    df = df[~df[orig_col].isna()]
    df[[orig_col, col]] = df[[col, orig_col]]   # swap: name -> col, original -> orig_col
    print(f"  {col} name resolved: {len(df):,} kept / {before - len(df):,} dropped")
    return df


def map_gene_synonyms(df, col='Tail'):
    """
    Normalise a gene column: apply synonym map -> canonical NCBI Symbol,
    annotate description, then swap canonical symbol into col.
   
    Uses the globally built synonym_map instead.
    """
    df = df.copy()
    alias_col  = f'{col}_Alias_mapped'
    detail_col = f'{col}_Detail_name'
    df[alias_col]  = df[col].apply(lambda x: synonym_map.get(x, x))
    df[detail_col] = df[alias_col].map(NCBI_ALL_GENEname_dict)
    df[[col, alias_col]] = df[[alias_col, col]]
    return df


def select_cols(df, desired):
    """Select only columns that exist in df from desired list."""
    return df[[c for c in desired if c in df.columns]]


def save(df, filename):
    """Save to OUT_PATH and print confirmation."""
    path = os.path.join(OUT_PATH, filename)
    df.to_csv(path, index=False)
    print(f"Saved {len(df):,} rows -> {path}")


print("Helper functions defined.")

Helper functions defined.


---
## 5 · Process and export each relation type

### 5a · Gene_Gene

In [18]:
Gene_Gene = BOCK[BOCK['Relation'] == 'Gene_Gene'].copy()

# Resolve ENSG node IDs to gene names, then normalise via synonym map
Gene_Gene = resolve_BOCK_name(Gene_Gene, 'Head')
Gene_Gene = resolve_BOCK_name(Gene_Gene, 'Tail')
Gene_Gene = map_gene_synonyms(Gene_Gene, col='Head')
Gene_Gene = map_gene_synonyms(Gene_Gene, col='Tail')
Gene_Gene['Head_ID_IS'] = 'NCBI_ID'
Gene_Gene['Tail_ID_IS'] = 'NCBI_ID'

print(f"Gene_Gene: {len(Gene_Gene):,} triples")
print("Relation_type values:", set(Gene_Gene['Relation_type']))
save(Gene_Gene, 'Gene_Gene_BOCK.csv')

  Head name resolved: 1,838,705 kept / 37 dropped
  Tail name resolved: 1,838,423 kept / 282 dropped
Gene_Gene: 1,838,423 triples
Relation_type values: {'seqSimilar', 'physInteracts', 'coexpresses'}
Saved 1,838,423 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/GPKG/Gene_Gene_GPKG.csv


### 5b · Gene_Phenotype

In [19]:
Gene_Phenotype = BOCK[BOCK['Relation'] == 'Gene_Phenotype'].copy()

Gene_Phenotype = resolve_BOCK_name(Gene_Phenotype, 'Head')
Gene_Phenotype = map_gene_synonyms(Gene_Phenotype, col='Head')
Gene_Phenotype['Head_ID_IS'] = 'NCBI_ID'

# Tail is an HPO ID — map to HPO term name
Gene_Phenotype['Tail_name']  = Gene_Phenotype['Tail'].map(phenotype_dict)
Gene_Phenotype['Tail_ID_IS'] = 'HPO_ID'

print(f"Gene_Phenotype: {len(Gene_Phenotype):,} triples")
save(Gene_Phenotype, 'Gene_Phenotype_BOCK.csv')

  Head name resolved: 209,416 kept / 0 dropped
Gene_Phenotype: 209,416 triples
Saved 209,416 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/GPKG/Gene_Phenotype_GPKG.csv


### 5c · Gene_BiologicalProcess

In [20]:
Gene_BiologicalProcess = BOCK[BOCK['Relation'] == 'Gene_BiologicalProcess'].copy()

Gene_BiologicalProcess = resolve_BOCK_name(Gene_BiologicalProcess, 'Head')
Gene_BiologicalProcess = map_gene_synonyms(Gene_BiologicalProcess, col='Head')
Gene_BiologicalProcess['Head_ID_IS'] = 'NCBI_ID'

# Tail is a GO ID — map to GO term name
Gene_BiologicalProcess['Tail_Detail_name'] = Gene_BiologicalProcess['Tail'].map(GO_dict)
Gene_BiologicalProcess['Tail_ID_IS']       = 'Quick_GO'

print(f"Gene_BiologicalProcess: {len(Gene_BiologicalProcess):,} triples")
save(Gene_BiologicalProcess, 'Gene_BiologicalProcess_BOCK.csv')

  Head name resolved: 93,395 kept / 281 dropped
Gene_BiologicalProcess: 93,395 triples
Saved 93,395 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/GPKG/Gene_BiologicalProcess_GPKG.csv


### 5d · Gene_CellularComponent

In [21]:
Gene_CellularComponent = BOCK[BOCK['Relation'] == 'Gene_CellularComponent'].copy()

Gene_CellularComponent = resolve_BOCK_name(Gene_CellularComponent, 'Head')
Gene_CellularComponent = map_gene_synonyms(Gene_CellularComponent, col='Head')
Gene_CellularComponent['Head_ID_IS'] = 'NCBI_ID'

Gene_CellularComponent['Tail_Detail_name'] = Gene_CellularComponent['Tail'].map(GO_dict)
Gene_CellularComponent['Tail_ID_IS']       = 'Quick_GO'

print(f"Gene_CellularComponent: {len(Gene_CellularComponent):,} triples")
save(Gene_CellularComponent, 'Gene_CellularComponent_BOCK.csv')

  Head name resolved: 58,263 kept / 169 dropped
Gene_CellularComponent: 58,263 triples
Saved 58,263 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/GPKG/Gene_CellularComponent_GPKG.csv


### 5e · Gene_MolecularFunction

In [22]:
Gene_MolecularFunction = BOCK[BOCK['Relation'] == 'Gene_MolecularFunction'].copy()

Gene_MolecularFunction = resolve_BOCK_name(Gene_MolecularFunction, 'Head')
Gene_MolecularFunction = map_gene_synonyms(Gene_MolecularFunction, col='Head')
Gene_MolecularFunction['Head_ID_IS'] = 'NCBI_ID'

Gene_MolecularFunction['Tail_Detail_name'] = Gene_MolecularFunction['Tail'].map(GO_dict)
# Drop rows where GO term name could not be resolved
before = len(Gene_MolecularFunction)
Gene_MolecularFunction = Gene_MolecularFunction[~Gene_MolecularFunction['Tail_Detail_name'].isna()]
print(f"GO filter: {len(Gene_MolecularFunction):,} kept / {before - len(Gene_MolecularFunction):,} dropped")
Gene_MolecularFunction['Tail_ID_IS'] = 'Quick_GO'

print(f"Gene_MolecularFunction: {len(Gene_MolecularFunction):,} triples")
save(Gene_MolecularFunction, 'Gene_MolecularFunction_BOCK.csv')

  Head name resolved: 43,147 kept / 184 dropped
GO filter: 43,147 kept / 0 dropped
Gene_MolecularFunction: 43,147 triples
Saved 43,147 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/GPKG/Gene_MolecularFunction_GPKG.csv


### 5f · Gene_ProteinComplex

In [23]:
Gene_ProteinComplex = BOCK[BOCK['Relation'] == 'Gene_ProteinComplex'].copy()

Gene_ProteinComplex = resolve_BOCK_name(Gene_ProteinComplex, 'Head')
Gene_ProteinComplex = map_gene_synonyms(Gene_ProteinComplex, col='Head')
Gene_ProteinComplex['Head_ID_IS'] = 'NCBI_ID'
# Tail is a protein complex ID — no external ontology mapping applied

print(f"Gene_ProteinComplex: {len(Gene_ProteinComplex):,} triples")
save(Gene_ProteinComplex, 'Gene_ProteinComplex_BOCK.csv')

  Head name resolved: 14,531 kept / 0 dropped
Gene_ProteinComplex: 14,531 triples
Saved 14,531 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/GPKG/Gene_ProteinComplex_GPKG.csv


### 5g · BiologicalProcess_BiologicalProcess

In [24]:
BiologicalProcess_BiologicalProcess = BOCK[BOCK['Relation'] == 'BiologicalProcess_BiologicalProcess'].copy()

BiologicalProcess_BiologicalProcess['Head_Detail_name'] = BiologicalProcess_BiologicalProcess['Head'].map(GO_dict)
before = len(BiologicalProcess_BiologicalProcess)
BiologicalProcess_BiologicalProcess = BiologicalProcess_BiologicalProcess[~BiologicalProcess_BiologicalProcess['Head_Detail_name'].isna()]
BiologicalProcess_BiologicalProcess['Head_ID_IS'] = 'Quick_GO'

BiologicalProcess_BiologicalProcess['Tail_Detail_name'] = BiologicalProcess_BiologicalProcess['Tail'].map(GO_dict)
BiologicalProcess_BiologicalProcess = BiologicalProcess_BiologicalProcess[~BiologicalProcess_BiologicalProcess['Tail_Detail_name'].isna()]
BiologicalProcess_BiologicalProcess['Tail_ID_IS'] = 'Quick_GO'

print(f"BiologicalProcess_BiologicalProcess: {len(BiologicalProcess_BiologicalProcess):,} triples")
save(BiologicalProcess_BiologicalProcess, 'BiologicalProcess_BiologicalProcess_BOCK.csv')

BiologicalProcess_BiologicalProcess: 33,102 triples
Saved 33,102 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/GPKG/BiologicalProcess_BiologicalProcess_GPKG.csv


### 5h · MolecularFunction_MolecularFunction

In [25]:
MolecularFunction_MolecularFunction = BOCK[BOCK['Relation'] == 'MolecularFunction_MolecularFunction'].copy()

MolecularFunction_MolecularFunction['Head_Detail_name'] = MolecularFunction_MolecularFunction['Head'].map(GO_dict)
MolecularFunction_MolecularFunction = MolecularFunction_MolecularFunction[~MolecularFunction_MolecularFunction['Head_Detail_name'].isna()]
MolecularFunction_MolecularFunction['Head_ID_IS'] = 'Quick_GO'

MolecularFunction_MolecularFunction['Tail_Detail_name'] = MolecularFunction_MolecularFunction['Tail'].map(GO_dict)
MolecularFunction_MolecularFunction = MolecularFunction_MolecularFunction[~MolecularFunction_MolecularFunction['Tail_Detail_name'].isna()]
MolecularFunction_MolecularFunction['Tail_ID_IS'] = 'Quick_GO'

print(f"MolecularFunction_MolecularFunction: {len(MolecularFunction_MolecularFunction):,} triples")
save(MolecularFunction_MolecularFunction, 'MolecularFunction_MolecularFunction_BOCK.csv')

MolecularFunction_MolecularFunction: 11,239 triples
Saved 11,239 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/GPKG/MolecularFunction_MolecularFunction_GPKG.csv


### 5i · CellularComponent_CellularComponent

In [26]:
CellularComponent_CellularComponent = BOCK[BOCK['Relation'] == 'CellularComponent_CellularComponent'].copy()

CellularComponent_CellularComponent['Head_Detail_name'] = CellularComponent_CellularComponent['Head'].map(GO_dict)
CellularComponent_CellularComponent = CellularComponent_CellularComponent[~CellularComponent_CellularComponent['Head_Detail_name'].isna()]
CellularComponent_CellularComponent['Head_ID_IS'] = 'Quick_GO'

CellularComponent_CellularComponent['Tail_Detail_name'] = CellularComponent_CellularComponent['Tail'].map(GO_dict)
CellularComponent_CellularComponent = CellularComponent_CellularComponent[~CellularComponent_CellularComponent['Tail_Detail_name'].isna()]
CellularComponent_CellularComponent['Tail_ID_IS'] = 'Quick_GO'

print(f"CellularComponent_CellularComponent: {len(CellularComponent_CellularComponent):,} triples")
save(CellularComponent_CellularComponent, 'CellularComponent_CellularComponent_BOCK.csv')

CellularComponent_CellularComponent: 793 triples
Saved 793 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/GPKG/CellularComponent_CellularComponent_GPKG.csv


### 5j · Phenotype_Phenotype

In [27]:
Phenotype_Phenotype = BOCK[BOCK['Relation'] == 'Phenotype_Phenotype'].copy()

Phenotype_Phenotype['Head_name'] = Phenotype_Phenotype['Head'].map(phenotype_dict)
Phenotype_Phenotype['Head_ID_IS'] = 'HPO_ID'

Phenotype_Phenotype['Tail_name'] = Phenotype_Phenotype['Tail'].map(phenotype_dict)
Phenotype_Phenotype['Tail_ID_IS'] = 'HPO_ID'

print(f"Phenotype_Phenotype: {len(Phenotype_Phenotype):,} triples")
save(Phenotype_Phenotype, 'Phenotype_Phenotype_BOCK.csv')

Phenotype_Phenotype: 16,000 triples
Saved 16,000 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/GPKG/Phenotype_Phenotype_GPKG.csv


### 5k · Disease_Phenotype

Head nodes use OMIM IDs (`mim:XXXXXX`) which are resolved to DOID via the DO xrefs crosswalk.  
Orphanet IDs (`orphanet:XXXXXX`) cannot be resolved via OMIM_dict and are dropped.

In [28]:
Disease_Phenotype = BOCK[BOCK['Relation'] == 'Disease_Phenotype'].copy()

# Resolve OMIM ID -> DOID (mim:XXXXXX -> DOID:XXXXXXX)
# Rows that resolve get DOID in Head; original mim: ID moves to Head_DOID (audit trail)
Disease_Phenotype['Head_DOID'] = Disease_Phenotype['Head'].map(OMIM_dict)
before = len(Disease_Phenotype)
Disease_Phenotype = Disease_Phenotype[~Disease_Phenotype['Head_DOID'].isna()]
print(f"OMIM -> DOID resolved: {len(Disease_Phenotype):,} kept / {before - len(Disease_Phenotype):,} dropped")
print("  (Dropped rows include Orphanet IDs and other non-OMIM disease identifiers)")

# Swap: DOID -> Head, original mim: ID -> Head_DOID
Disease_Phenotype[['Head_DOID', 'Head']] = Disease_Phenotype[['Head', 'Head_DOID']]

Disease_Phenotype['Tail_name']  = Disease_Phenotype['Tail'].map(phenotype_dict)
before = len(Disease_Phenotype)
Disease_Phenotype = Disease_Phenotype[~Disease_Phenotype['Tail_name'].isna()]
print(f"HPO Tail resolved: {len(Disease_Phenotype):,} kept / {before - len(Disease_Phenotype):,} dropped")

Disease_Phenotype['Head_ID_IS'] = 'DOID'
Disease_Phenotype['Tail_ID_IS'] = 'HP_ID'

print(f"Disease_Phenotype: {len(Disease_Phenotype):,} triples")
print("Head ID prefixes:", set(Disease_Phenotype['Head'].str[:4]))
save(Disease_Phenotype, 'Disease_Phenotype_BOCK.csv')

OMIM -> DOID resolved: 80,347 kept / 152,534 dropped
  (Dropped rows include Orphanet IDs and other non-OMIM disease identifiers)
HPO Tail resolved: 80,347 kept / 0 dropped
Disease_Phenotype: 80,347 triples
Head ID prefixes: {'DOID'}
Saved 80,347 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/GPKG/Disease_Phenotype_GPKG.csv


---
## 6 · Dropped relation types (commented out in original)

In [29]:
# Gene_ProteinFamily and Gene_ProteinDomain were commented out in the original notebook.
# They are intentionally not processed here.
# To enable them, uncomment the blocks below.

# Gene_ProteinFamily = BOCK[BOCK['Relation'] == 'Gene_ProteinFamily'].copy()
# Gene_ProteinFamily = resolve_BOCK_name(Gene_ProteinFamily, 'Head')
# Gene_ProteinFamily = map_gene_synonyms(Gene_ProteinFamily, col='Head')
# Gene_ProteinFamily['Head_ID_IS'] = 'NCBI_ID'
# save(Gene_ProteinFamily, 'Gene_ProteinFamily_BOCK.csv')

# Gene_ProteinDomain = BOCK[BOCK['Relation'] == 'Gene_ProteinDomain'].copy()
# Gene_ProteinDomain = resolve_BOCK_name(Gene_ProteinDomain, 'Head')
# Gene_ProteinDomain = map_gene_synonyms(Gene_ProteinDomain, col='Head')
# Gene_ProteinDomain['Head_ID_IS'] = 'NCBI_ID'
# save(Gene_ProteinDomain, 'Gene_ProteinDomain_BOCK.csv')

print("Gene_ProteinFamily and Gene_ProteinDomain: DROPPED (not in scope)")

Gene_ProteinFamily and Gene_ProteinDomain: DROPPED (not in scope)


---
## 7 · Summary — all output files

In [3]:
# Updated to use OUT_PATH.
print(f"Output files under: {OUT_PATH}\n")
cols = ["Relation","Head_type","Tail_type","Source","KG_Source","Head_ID_IS","Tail_ID_IS"]
df_list = []

for file_path in sorted(glob(os.path.join(OUT_PATH, '*.csv'))):
    try:
        total_df   = pd.read_csv(file_path)
        total_rows = len(total_df)
        temp_df    = total_df.head(5)
        available  = [c for c in cols if c in temp_df.columns]
        row = {'Filename': os.path.basename(file_path), 'no. of triples': total_rows}
        for c in available:
            row[c] = set(temp_df[c].dropna().tolist())
        df_list.append(pd.DataFrame([row]))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

combined_df = pd.concat(df_list, ignore_index=True)
print(f"Total files: {len(combined_df)}")
combined_df

Output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/BOCK/

Total files: 11


,Filename,no. of triples,Relation,Head_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS
0,BiologicalProcess_BiologicalProcess_GPKG.csv,33102,{BiologicalProcess_BiologicalProcess},{BiologicalProcess},{BiologicalProcess},{GPKG},{Quick_GO},{Quick_GO}
1,CellularComponent_CellularComponent_GPKG.csv,793,{CellularComponent_CellularComponent},{CellularComponent},{CellularComponent},{GPKG},{Quick_GO},{Quick_GO}
2,Disease_Phenotype_GPKG.csv,80347,{Disease_Phenotype},{Disease},{Phenotype},{GPKG},{DOID},{HP_ID}
3,Gene_BiologicalProcess_GPKG.csv,93395,{Gene_BiologicalProcess},{Gene},{BiologicalProcess},{GPKG},{NCBI_ID},{Quick_GO}
4,Gene_CellularComponent_GPKG.csv,58263,{Gene_CellularComponent},{Gene},{CellularComponent},{GPKG},{NCBI_ID},{Quick_GO}
5,Gene_Gene_GPKG.csv,1838423,{Gene_Gene},{Gene},{Gene},{GPKG},{NCBI_ID},{NCBI_ID}
6,Gene_MolecularFunction_GPKG.csv,43147,{Gene_MolecularFunction},{Gene},{MolecularFunction},{GPKG},{NCBI_ID},{Quick_GO}
7,Gene_Phenotype_GPKG.csv,209416,{Gene_Phenotype},{Gene},{Phenotype},{GPKG},{NCBI_ID},{HPO_ID}
8,Gene_ProteinComplex_GPKG.csv,14531,{Gene_ProteinComplex},{Gene},{ProteinComplex},{GPKG},{NCBI_ID},NaN
9,MolecularFunction_MolecularFunction_GPKG.csv,11239,{MolecularFunction_MolecularFunction},{MolecularFunction},{MolecularFunction},{GPKG},{Quick_GO},{Quick_GO}


In [6]:
# Updated to use OUT_PATH.
print(f"Output files under: {OUT_PATH}\n")
cols = ["Relation","Head_type","Tail_type","Source","KG_Source","Head_ID_IS","Tail_ID_IS"]
df_list = []

for file_path in sorted(glob(os.path.join(OUT_PATH, '*.csv'))):
    try:
        total_df   = pd.read_csv(file_path)
        total_rows = len(total_df)
        temp_df    = total_df.head(5)
        available  = [c for c in cols if c in temp_df.columns]
        row = {'Filename': os.path.basename(file_path), 'no. of triples': total_rows}
        for c in available:
            row[c] = set(temp_df[c].dropna().tolist())
        df_list.append(pd.DataFrame([row]))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

combined_df = pd.concat(df_list, ignore_index=True)
print(f"Total files: {len(combined_df)}")
combined_df

Output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/BOCK/

Total files: 11


,Filename,no. of triples,Relation,Head_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS
0,BiologicalProcess_BiologicalProcess_BOCK.csv,33102,{BiologicalProcess_BiologicalProcess},{BiologicalProcess},{BiologicalProcess},{BOCK},{Quick_GO},{Quick_GO}
1,CellularComponent_CellularComponent_BOCK.csv,793,{CellularComponent_CellularComponent},{CellularComponent},{CellularComponent},{BOCK},{Quick_GO},{Quick_GO}
2,Disease_Phenotype_BOCK.csv,80347,{Disease_Phenotype},{Disease},{Phenotype},{BOCK},{DOID},{HP_ID}
3,Gene_BiologicalProcess_BOCK.csv,93395,{Gene_BiologicalProcess},{Gene},{BiologicalProcess},{BOCK},{NCBI_ID},{Quick_GO}
4,Gene_CellularComponent_BOCK.csv,58263,{Gene_CellularComponent},{Gene},{CellularComponent},{BOCK},{NCBI_ID},{Quick_GO}
5,Gene_Gene_BOCK.csv,1838423,{Gene_Gene},{Gene},{Gene},{BOCK},{NCBI_ID},{NCBI_ID}
6,Gene_MolecularFunction_BOCK.csv,43147,{Gene_MolecularFunction},{Gene},{MolecularFunction},{BOCK},{NCBI_ID},{Quick_GO}
7,Gene_Phenotype_BOCK.csv,209416,{Gene_Phenotype},{Gene},{Phenotype},{BOCK},{NCBI_ID},{HPO_ID}
8,Gene_ProteinComplex_BOCK.csv,14531,{Gene_ProteinComplex},{Gene},{ProteinComplex},{BOCK},{NCBI_ID},NaN
9,MolecularFunction_MolecularFunction_BOCK.csv,11239,{MolecularFunction_MolecularFunction},{MolecularFunction},{MolecularFunction},{BOCK},{Quick_GO},{Quick_GO}
